<a href="https://colab.research.google.com/github/usami0jp/ai2023-old/blob/main/stylegan6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# -*- coding: utf-8 -*-
"""StyleGAN_colab.ipynb

Automatically generated by Colab.

Original file is located at
    https://colab.research.google.com/drive/1ETQ8lxfCApKeow87hVLQl11sZomJLWd_

# StyleGAN

GANで高解像な画像生成をするためには，人の目で見ても区別が付かない程，鮮明な画像を生成する必要があります．
つまり，鮮明でない画像は，Discriminatorで容易に本物か偽物かを区別できてしまうため，モード崩壊が起こります．<br>
この問題を回避する方法として，徐々に層を追加するプログレッシブな学習プロセスを使用しています．プログレッシブな学習は，Progressive Growing GAN (PG-GAN)で提案された方法です．

PG-GANと異なる点は，潜在変数を直接画像生成に使用するのではなく，Adaptive Instance Normalization (AdaIN)のパラメータに利用しています．
ここで，AdaINは，しばしばスタイル変換に利用されるNormalization方法です．
潜在変数をAdaINのパラメータに利用することでネットワークの浅い層と深い層で変化するスタイルに違いがあることを発見しました．

Generatorの入力層へ与える変数は，ランダムサンプリングしたベクトルではなく，学習初期に決定した任意のサイズの定数（4x4のmap）を入力として学習をしています．

---
### このノートブックの構成
1. **StyleGAN のネットワーク構造**（PyTorch による実装）… 各レイヤを定義し，小さな設定で動作を確認します．
2. **StyleGAN2 での改良点** … AdaIN が Weight demodulation に置き換わった理由を，自分で書いた実装と公式実装を比較しながら確認します．
3. **学習済みモデルによる画像生成** … NVIDIA 公式の学習済み StyleGAN2（CelebA-HQ / FFHQ など）を使って，
   Truncation trick，潜在空間 $\mathcal{Z}$ と $\mathcal{W}$ の補間，スタイルミキシング，ノイズの効果を実験します．

> **準備**：メニューの「ランタイム」→「ランタイムのタイプを変更」で **GPU（T4 など）** を選択してから実行してください．

## 必要モジュールのインポート
"""


"""# 1. StyleGAN のネットワーク構造

## ネットワーク構築のために必要なレイヤの定義
### Equalized Convolution，Equalized Linear
一般的にDeep Learningで識別，認識，分類をするときにネットワークを初期化する方法として，Heの初期化やXavierなど正規分布に基づく初期化が多用されています．
しかしながら，これらの初期化方法とAdamやRMSPropなどの最適化関数を組み合わせて学習をすることは，画像生成には適さないことがPG-GANの論文中で示されています [2]．
StyleGANでも同様のレイヤが用いられており，以下のように動的に初期値を決定しています．
$$
w^{\prime}\leftarrow w\sqrt{\frac{2}{Ch * K}}
$$
ここで，$w$は正規分布からサンプリングした重みフィルタ，$Ch$はチャネル数，$K$はカーネルサイズ（全結合の場合は1）です．
"""


"""### FusedUpsampling，FusedDownsampling
DC-GANをはじめとする多くのGANで問題視されている1つが，Generatorのアップサンプル時にartifactが載りやすいことです．
アップサンプリングでは，bilinear補間やdeconvolutionを用いて引き延ばすため，artifactが載りやすくなります．
この問題を解消するために，アップサンプル時のConvolutionで用いる重みフィルタにローパスフィルタ（ガウスフィルタ）を用いています．
これによって，鮮明な画像生成の手助けをしています．

この問題はダウンサンプル時にも同様のことが言えて，ダウンサンプル前にガウスフィルタを用います．
"""


"""### Pixel-wise normalization，Adaptive instance normalization
DC-GANでは，Batch Normalizationをネットワーク全体に使用することで高解像な画像生成をしていました．
Batch Normalizationは高精度な認識を得るためのネットワークに使用することは効果的ですが，単純にミニバッチで正規化をする処理は画像生成には適さないことが判明しました．
Batch Normalizationは，学習可能なパラメータを含むので，画像生成には不要かつモード崩壊を起こす1つの原因となるため，排除してPixel-wise normalizationと呼ばれる正規化が利用されています．
これはPG-GANでGeneratorの全層に用いられている，各画像で正規化をする方法です．
StyleGANでは，Mapping networkのみに利用されています．Pixel-wise normalizationは以下の式のとおりです．
$$
b_{x, y} = \frac{a_{x, y}}{\sqrt{\frac{1}{N}\sum_{j=0}^{N-1}(a_{x,y}^{j})^{2}+\epsilon}}
$$
ここで，$a_{x,y}, b_{x,y}$は，それぞれ正規化前の特徴マップと正規化後の特徴マップです．また，Nは特徴マップの総数で，$\epsilon=10^{-8}$です．

Adaptive instance normalizationは，pytorch内部で実装がされていますが，内部処理を少々変更する必要があるので以下に定義してあります．
AdaINは，特徴マップ $x$ をチャネルごとに平均0・分散1に正規化した後，スタイル $y=(\gamma, \beta)$ でスケールとバイアスを与えます．
$$
\mathrm{AdaIN}(x_i, y) = \gamma_i \frac{x_i - \mu(x_i)}{\sigma(x_i)} + \beta_i
$$
"""

class AdaptiveInstanceNorm(nn.Module):
    '''
    Adaptive instance normalization
    '''
    def __init__(self, in_channel, style_dim):
        super().__init__()

        self.norm = nn.InstanceNorm2d(in_channel)
        self.style = EqualLinear(style_dim, in_channel * 2)

        self.style.linear.bias.data[:in_channel] = 1
        self.style.linear.bias.data[in_channel:] = 0

    def forward(self, input, style):
        style = self.style(style).unsqueeze(2).unsqueeze(3)
        gamma, beta = style.chunk(2, 1)

        out = self.norm(input)
        out = gamma * out + beta

        return out

"""**確認**：AdaIN を通すと，入力の特徴マップの統計量は消え，各チャネルの平均が $\beta$，標準偏差が $|\gamma|$ に置き換わります．
つまり「特徴マップの統計量 = スタイル」をスタイルコードで丸ごと上書きしている，

"""### その他のレイヤ
* ConstantInput：Generatorに入力する任意のサイズの定数マップを作成するためのクラス．
* NoiseInjection：GeneratorのAdaINの前に与えるノイズを作成するために必要なクラス．
* Blur：ダウンサンプル，アップサンプル前後でローパスフィルタを施すために必要となるクラス．
"""


"""## Generatorの定義
Generatorはいくつかのブロックを用いて構成します．
各ブロックは，2層の畳み込み処理を含んでいます．
この2層の内，１つ目の畳み込み処理の前に特徴マップをアップサンプルします．
全てのブロックの後に使用する活性化関数は，LeakyReLUです．
活性化関数の前にノイズを加算する処理であるNoise injectionを導入します．
活性化関数を通した後は，mapping networkで写像した空間$\mathcal{W}$からサンプリングした潜在変数と特徴マップを用いてAdaptive instance normalizationで正規化します．Generatorは，mapping networkを必要とするため，全結合層8層とLeakyReLUで非線形変換するネットワークを構築しています．


また，StyleGANはプログレッシブな学習をするため，高解像度の画像を生成するために層を追加．
しかしながら，数epoch学習した層と次の解像度を生成する初期値の畳み込み処理は，ギャップが大きいため徐々に新たに追加した層を反映するように線形変化する重み$\alpha$を使用します．<br>
以下に16×16から32×32への移り変わりを示す．


"""## Discriminatorの定義
DiscriminatorもGeneratorと同様でいくつかのブロックが積み重なったネットワーク．DiscriminatorへはBatchNormalizationなどの正規化を一切使用しません．
DiscriminatorもGeneratorと同様で，スムーズに解像度の移り変わりができるように線形増加する$\alpha$によって徐々に次の解像度へ移り変わるネットワーク構造を使用します．

<img src="https://dl.dropboxusercontent.com/s/zpfdnq4gcilochs/StyleGAN_D_transition.png" width=40%>
"

"""## 動作確認：プログレッシブな成長
小さめの設定（チャネル数を減らしたもの）でGeneratorとDiscriminatorを作り，`depth` を増やすと出力解像度が 4→8→16→… と倍になっていく様子を確認します．
`alpha < 1` のときは，1つ前の解像度の出力（を拡大したもの）と新しく追加した層の出力を $\alpha$ で混ぜている．

※ ここでは学習していないランダムな重みなので，生成される画像はノイズ．StyleGANの学習には大量の計算資源（GPU数枚で数日〜数週間）が必要なため，Colab上での学習は行わず，次章以降で学習済みモデルを使います．
"""

"""# 2. StyleGAN2 での改良点

StyleGAN の生成画像をよく見ると，水滴のような染み（**droplet artifact**）が現れる．
StyleGAN2 [4] では，この原因が **AdaIN の正規化** にあると突き止めました．
AdaIN はチャネルごとの平均・分散を消してしまうため，Generator は「一部に極端に大きな値の染みを作って，正規化後の他の値の大きさを調整する」というズルを覚えてしまうのです．


そこで StyleGAN2 では，次のように変更．

| | StyleGAN | StyleGAN2 |
|---|---|---|
| スタイルの与え方 | AdaIN（特徴マップを正規化してから $\gamma, \beta$ を掛ける） | **Weight modulation / demodulation**（畳み込みの重みにスタイルを掛ける） |
| 高解像度化 | Progressive growing（層を徐々に追加） | 最初から全層で学習（skip接続 / residual接続） |
| ノイズ・バイアス | AdaIN の前 | スタイルの外側に移動 |
| 正則化 | — | Path length regularization，lazy R1 正則化 |


### Weight modulation / demodulation
特徴マップではなく **畳み込みの重み** $w_{ijk}$（$i$：入力チャネル，$j$：出力チャネル，$k$：カーネル内の位置）にスタイル $s_i$ を掛けます（modulation）．
$$
w'_{ijk} = s_i \cdot w_{ijk}
$$
その後，出力の標準偏差が1になると仮定して重みを正規化します（demodulation）．
$$
w''_{ijk} = \frac{w'_{ijk}}{\sqrt{\sum_{i,k} {w'_{ijk}}^2 + \epsilon}}
$$
AdaIN では「実際の特徴マップの統計量」で割っていたのに対し，demodulation では「重みから期待される統計量」で割るので，特徴マップの中身に依存しません．これにより染みを作る抜け道がなくなりました．

以下では，NVIDIA 公式の実装（[NVlabs/stylegan3](https://github.com/NVlabs/stylegan3)．StyleGAN2 のコードも含まれています）をダウンロードし，自分で書いた modulated convolution と比較してみます．

## 公式実装のダウンロードと準備
最近の Colab（Python 3.12 以降）では，公式実装が使っている `pkg_resources`・`distutils` が使えないことがあるため，その部分だけを書き換えています．

`USE_CUSTOM_CUDA_OPS` を `True` にすると，公式の高速な CUDA カーネルをコンパイルして使います（初回に数分かかり，環境によっては失敗します）．`False` なら PyTorch だけで書かれた実装を使います．画像生成だけなら `False` で十分です．


"""## 自分で modulated convolution を書いて公式実装と比べる
バッチ内の各サンプルで異なる重みを使う必要があるので，バッチを「グループ」とみなして `groups=N` のグループ畳み込みで一度に計算するのがポイントです．
"""

def my_modulated_conv2d(x, weight, styles, demodulate=True, padding=1):
    '''
    x      : 入力特徴マップ [N, Cin, H, W]
    weight : 畳み込みの重み [Cout, Cin, k, k]
    styles : スタイル s     [N, Cin]   （w からアフィン変換で求めたもの）
    '''
    N, Cin, H, W = x.shape
    Cout, _, k, _ = weight.shape

    # (1) modulation : 入力チャネルごとにスタイルを掛ける    w' = s_i * w
    w = weight[None] * styles[:, None, :, None, None]           # [N, Cout, Cin, k, k]

    # (2) demodulation : 出力チャネルごとに正規化            w'' = w' / sqrt(sum w'^2)
    if demodulate:
        d = (w.square().sum(dim=[2, 3, 4]) + 1e-8).rsqrt()      # [N, Cout]
        w = w * d[:, :, None, None, None]

    # (3) グループ畳み込みで，サンプルごとに異なる重みで畳み込む
    x = x.reshape(1, N * Cin, H, W)
    w = w.reshape(N * Cout, Cin, k, k)
    out = F.conv2d(x, w, padding=padding, groups=N)
    return out.reshape(N, Cout, H, W)


torch.manual_seed(0)
x = torch.randn(4, 16, 8, 8, device=device)
weight = torch.randn(32, 16, 3, 3, device=device)
styles = torch.randn(4, 16, device=device)

out_mine = my_modulated_conv2d(x, weight, styles)
out_official = networks_stylegan2.modulated_conv2d(x=x, weight=weight, styles=styles, padding=1)
print('出力サイズ        :', tuple(out_mine.shape))
print('公式実装との最大誤差:', (out_mine - out_official).abs().max().item())
print('出力の標準偏差     :', out_mine.std().item(), '（demodulation により約1になる）')

"""公式実装の中身（`fused_modconv=True` の分岐が上の実装に相当）．"""

src = open(networks_stylegan2.__file__, encoding='utf-8').read()
start = src.index('def modulated_conv2d(')
print(src[start:src.index('#----', start)])

"""# 3. 学習済みモデルによる画像生成

NVIDIA が公開している学習済みの StyleGAN2 モデルを読み込む．`MODEL` から好きなモデルを選ぶ．

| モデル | 内容 | 解像度 |
|---|---|---|
| `stylegan2-celebahq-256x256.pkl` | 有名人の顔（CelebA-HQ） | 256 |
| `stylegan2-ffhq-256x256.pkl` / `-512x512` / `-1024x1024` | 人の顔（FFHQ） | 256〜1024 |
| `stylegan2-afhqv2-512x512.pkl` | 動物の顔（犬・猫・野生動物） | 512 |
| `stylegan2-afhqcat-512x512.pkl` / `afhqdog` | 猫 / 犬 | 512 |
| `stylegan2-metfaces-1024x1024.pkl` | 美術館の肖像画（MetFaces） | 1024 |
| `stylegan2-lsundog-256x256.pkl` | 犬（LSUN） | 256 |

※ モデルは NVIDIA Source Code License（非商用利用のみ）で公開．
"""

MODEL = 'stylegan2-celebahq-256x256.pkl'  #@param ['stylegan2-celebahq-256x256.pkl', 'stylegan2-ffhq-256x256.pkl', 'stylegan2-ffhq-512x512.pkl', 'stylegan2-ffhq-1024x1024.pkl', 'stylegan2-afhqv2-512x512.pkl', 'stylegan2-afhqcat-512x512.pkl', 'stylegan2-afhqdog-512x512.pkl', 'stylegan2-metfaces-1024x1024.pkl', 'stylegan2-lsundog-256x256.pkl']

url = 'https://api.ngc.nvidia.com/v2/models/nvidia/research/stylegan2/versions/1/files/' + MODEL



"""### ネットワークの構造を確認
Generator は **mapping network**（$z \to w$，全結合8層）と **synthesis network**（$w$ から画像を作る畳み込み部分）に分かれています．
synthesis network の各層がどの番号の $w$ を受け取るかを表示．
解像度の低い層（4×4，8×8）が顔の向きや輪郭などの大まかな構造，高い層が色や細かい質感を担当．
"""


"""## 画像の生成
乱数のシード（seed）から潜在変数 $z$ を作り，画像を生成します．シードを変えると別の画像になります．
"""
"""## Truncation trick
$\mathcal{W}$ 空間の平均 $\bar{w}$ に向かって $w$ を縮めることで，多様性と引き換えに画質を安定させるテクニックです．
$$
w' = \bar{w} + \psi (w - \bar{w})
$$
$\psi = 0$ のときは常に「平均的な顔」になり，$\psi$ を大きくするほど個性的（ただし破綻もしやすい）な画像になります．
"""


"""## 潜在空間の補間：$\mathcal{Z}$ と $\mathcal{W}$ の比較
2つの画像の間を補間してみます．
StyleGAN では mapping network によって **もつれの少ない（disentangled）** 空間 $\mathcal{W}$ が得られるため，
$\mathcal{Z}$ 上で補間するよりも $\mathcal{W}$ 上で補間した方が，変化が滑らかになると言われています．
"""


"""アニメーションでも確認．"""



"""## スタイルミキシング
StyleGAN の最大の特徴は，**synthesis network の層ごとに異なる $w$ を与えられる** ことです．
画像Aの $w$ のうち，ある解像度の層に与える部分だけを画像Bの $w$ に入れ替えると，Bの特徴のうちその解像度が担当するものだけがAに移ります．

* **Coarse（4×4〜8×8）**：顔の向き，輪郭，髪型など大まかな構造
* **Middle（16×16〜32×32）**：顔のパーツ，目の開き具合，表情など
* **Fine（64×64〜）**：肌や髪の色，照明などの細かい特徴

下の図では，各行の左端が画像A（ソース），上段が画像B（スタイルを借りる側）です．
"""

src_seeds = [30, 31, 32]       # 画像A（行）
dst_seeds = [40, 41, 42, 43]   # 画像B（列）
psi = 0.7

# 解像度ごとの w の範囲（上の対応表を参照）
mix_ranges = {
    'Coarse (4〜8)':  range(0, 4),
    'Middle (16〜32)': range(4, 8),
    'Fine (64〜)':    range(8, G.num_ws),
}

"""## 解像度ごとの補間
スタイルミキシングを連続的に行うと，特定の解像度の特徴だけを A から B へ徐々に変化．
"""


"""## ノイズの効果（Stochastic variation）
各層には $w$ とは別にランダムなノイズが加えられる．
同じ $w$ のままノイズだけを変えると，顔の人物は同じまま，髪の毛の流れや肌のシワといった **確率的な細部** だけが変化します．
右端は，ノイズを変えて100回生成したときの画素ごとの標準偏差（明るいところほどノイズの影響を受ける）．
"""

seed = 60  #@param

